## Maria Alexeenko 337982318 , Ilay Sabach 315377788 , Victoriaa Golovitsky 211491980

In [1]:
!pip install spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 100.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [4]:
import zipfile
import json
import pandas as pd
from collections import Counter
import spacy
import os

zip_file_path = 'G08-20251118.zip'

if os.path.exists(zip_file_path):
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall('.')

    for file in os.listdir('.'):
        if file.endswith('.jsonl'):
            print(f"- {file}")
else:
    print(f"File/s not found")

- qrel_8.jsonl
- all_docs_8.jsonl
- queries_8.jsonl


## Question 1

In [5]:
nlp = spacy.load("en_core_web_sm")

docs = []
# Open the JSONL file, where each line represents a separate document in JSON format
with open('all_docs_8.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        # Parse the JSON object from the current line and append it to the document list
        docs.append(json.loads(line))

# Extract the 'title' and 'abstract' fields from each document separately
titles = [d['title'] for d in docs]
abstracts = [d['abstract'] for d in docs]

print(f"Loaded {len(docs)} documents.")

Loaded 3000 documents.


In [7]:
import numpy as np

def analyze_text_basic(text_list, field_name):
    all_tokens = []
    unique_counts_per_doc = []

    # Tokenize each text and collect tokens and unique-word counts
    for doc in nlp.pipe(text_list, disable=["ner", "parser"]):
        tokens = [token.text for token in doc]

        all_tokens.extend(tokens)
        unique_counts_per_doc.append(len(set(tokens)))

    # Compute basic statistics required by the assignment
    total_unique_words = len(set(all_tokens))
    avg_unique = np.mean(unique_counts_per_doc)
    word_freq = Counter(all_tokens)

    # Print results for the current text field
    print(f"--- Analysis for: {field_name} ---")
    print(f"Total Unique Words: {total_unique_words}")
    print(f"Average Unique Words per Doc: {avg_unique:.2f}")

    # Top 20 most frequent tokens
    print("\nTop 20 Most Common:")
    print(word_freq.most_common(20))

    # 20 rarest tokens (sample)
    print("\nTop 20 Rarest (Sample):")
    print(word_freq.most_common()[:-21:-1])

# Run the analysis for both fields separately
print("\nTop Most Common Titles:")
analyze_text_basic(titles, "Titles - Raw")
print("\nTop Most Common Abstract:")
analyze_text_basic(abstracts, "Abstracts - Raw")


Top Most Common Titles:
--- Analysis for: Titles - Raw ---
Total Unique Words: 9552
Average Unique Words per Doc: 14.44

Top 20 Most Common:
[('.', 2840), ('of', 2522), ('in', 1557), ('-', 1479), ('and', 1459), ('the', 1276), (',', 635), ('a', 626), (':', 544), ('with', 449), ('for', 423), ('to', 317), ('[', 280), (']', 280), ('protein', 274), ('by', 254), ('The', 239), ('gene', 197), ('on', 193), (')', 182)]

Top 20 Rarest (Sample):
[('platform', 1), ('Infectious', 1), ('Bioactive', 1), ('gamete', 1), ('agglutination', 1), ('proportions', 1), ('Follistatin', 1), ('losartan', 1), ('Cataract', 1), ('cataracts', 1), ('subcapsular', 1), ('posterior', 1), ('bars', 1), ('Instruments', 1), ('oligomerization', 1), ('disassembly', 1), ('Microtubule', 1), ('CreB', 1), ('deubiquitinating', 1), ('stabilizes', 1)]

Top Most Common Abstract:
--- Analysis for: Abstracts - Raw ---
Total Unique Words: 31386
Average Unique Words per Doc: 99.19

Top 20 Most Common:
[('the', 22905), ('of', 21979), (',',

In the initial phase, we utilized the spaCy NLP library to tokenize the raw text of both the titles and abstracts. We separated the text into individual tokens without applying any filtering. As expected, the analysis of the most common words revealed that the list is dominated by 'stop-words'- grammatical connectors such as 'the', 'of', 'and', and 'in'. These words appear with high frequency but carry very little semantic meaning regarding the specific medical content of the documents. This indicates that raw frequency analysis is insufficient for understanding the core topics of the corpus without further processing.

## Question 2

In [8]:
def analyze_text_cleaned(text_list, field_name):
    all_tokens = []
    unique_counts_per_doc = []

    # Tokenize and remove stopwords, punctuation, and URLs
    for doc in nlp.pipe(text_list, disable=["ner", "parser"]):
        tokens = [
            token.text.lower() for token in doc
            if not token.is_stop and not token.is_punct and not token.like_url and token.is_alpha
        ]
        all_tokens.extend(tokens)
        unique_counts_per_doc.append(len(set(tokens)))

    # Compute required statistics after cleaning
    total_unique_words = len(set(all_tokens))
    avg_unique = np.mean(unique_counts_per_doc)
    word_freq = Counter(all_tokens)

    # Display main insights for the cleaned text
    print(f"--- Analysis for: {field_name} (CLEANED) ---")
    print(f"Total Unique Words: {total_unique_words}")
    print(f"Average Unique Words per Doc: {avg_unique:.2f}")
    print(word_freq.most_common(20))


# Run cleaned analysis for both fields
print("\nTop 20 Most Common Titles:")
analyze_text_cleaned(titles, "Titles - Cleaned")

print("\nTop 20 Most Common Abstract:")
analyze_text_cleaned(abstracts, "Abstracts - Cleaned")



Top 20 Most Common Titles:
--- Analysis for: Titles - Cleaned (CLEANED) ---
Total Unique Words: 7471
Average Unique Words per Doc: 8.70
[('protein', 280), ('gene', 206), ('human', 177), ('factor', 171), ('cells', 151), ('disease', 147), ('cell', 138), ('patients', 138), ('expression', 132), ('risk', 127), ('alpha', 116), ('dna', 108), ('study', 107), ('stroke', 107), ('rat', 105), ('proteins', 104), ('apolipoprotein', 95), ('v', 83), ('analysis', 79), ('associated', 78)]

Top 20 Most Common Abstract:
--- Analysis for: Abstracts - Cleaned (CLEANED) ---
Total Unique Words: 21136
Average Unique Words per Doc: 61.17
[('patients', 2090), ('protein', 1792), ('cells', 1732), ('cell', 1124), ('proteins', 1122), ('gene', 1099), ('results', 1070), ('risk', 969), ('study', 887), ('factor', 823), ('associated', 786), ('p', 763), ('disease', 750), ('dna', 726), ('activity', 725), ('expression', 715), ('found', 694), ('levels', 664), ('stroke', 663), ('group', 651)]


We performed a cleaning process by filtering out stop-words, punctuation marks, and URLs using spaCy's built-in attributes (is_stop, is_punct, like_url, is_alpha). We also converted all tokens to lowercase to ensure consistency.

There is a significant change in the top-20 list compared to the previous step. The meaningless grammatical words have been replaced by domain-specific terms such as 'patients', 'cells', 'treatment', 'study', and 'protein'. This new list provides a clear insight into the nature of the corpus, confirming that the documents deal with medical research, clinical trials, and biological processes. The vocabulary size decreased as 'noise' was removed, increasing the signal-to-noise ratio of our data.

## Question 3

In [9]:
def analyze_text_lemmatized(text_list, field_name):
    all_tokens = []

    # Tokenize and keep only meaningful lemmas
    for doc in nlp.pipe(text_list, disable=["ner", "parser"]):
        tokens = [
            token.lemma_.lower()
            for token in doc
            if not token.is_stop           # remove stopwords
            and not token.is_punct         # remove punctuation
            and not token.like_url         # remove URLs
            and token.text.strip() != ""   # skip empty tokens
        ]
        all_tokens.extend(tokens)

    # Count lemma frequency across corpus
    word_freq = Counter(all_tokens)

    # Display the most common lemmas
    print(f"--- Analysis for: {field_name} (LEMMATIZED) ---")
    print(word_freq.most_common(20))


# Run lemma-based analysis on both fields
print("\nTop 20 Most Common Abstracts:")
analyze_text_lemmatized(abstracts, "Abstracts - Lemmatized")

print("\nTop 20 Most Common Titles:")
analyze_text_lemmatized(titles, "Titles - Lemmatized")


Top 20 Most Common Abstracts:
--- Analysis for: Abstracts - Lemmatized (LEMMATIZED) ---
[('protein', 2912), ('cell', 2856), ('patient', 2372), ('study', 1582), ('gene', 1474), ('result', 1425), ('factor', 1350), ('increase', 1073), ('risk', 1003), ('group', 965), ('control', 960), ('level', 956), ('high', 921), ('activity', 887), ('disease', 853), ('mutation', 846), ('show', 836), ('associate', 833), ('suggest', 829), ('bind', 826)]

Top 20 Most Common Titles:
--- Analysis for: Titles - Lemmatized (LEMMATIZED) ---
[('protein', 383), ('cell', 289), ('gene', 237), ('factor', 231), ('human', 188), ('disease', 161), ('patient', 159), ('study', 144), ('rat', 136), ('expression', 132), ('risk', 128), ('effect', 122), ('alpha', 116), ('mouse', 112), ('dna', 108), ('stroke', 108), ('receptor', 101), ('mutation', 101), ('apolipoprotein', 99), ('analysis', 84)]


To further optimize the vocabulary, we applied Lemmatization. This process maps different inflected forms of a word to their common base form (lemma). For example, the words 'studies' and 'study' were unified under 'study', and 'cells' became 'cell'.

Method- We accessed the .lemma_ attribute of each token in spaCy pipeline. Result: This step successfully reduced the redundancy in our vocabulary. Consequently, the frequency counts for the base forms increased, providing a more accurate representation of the concepts mentioned in the text. This allows the search engine to match a query for 'tumor' with a document containing 'tumors', significantly improving the retrieval logic.

## Question 4

In [11]:
!pip install nltk
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [12]:
from nltk.corpus import wordnet

nlp = spacy.load("en_core_web_sm")

def get_canonical_word(word):
    """
    Returns a normalized synonym based on WordNet.
    If no synonym exists, returns the original word.
    """
    synsets = wordnet.synsets(word)
    if synsets:
        canonical = synsets[0].lemmas()[0].name()
        return canonical.replace('_', ' ').lower()
    return word

def analyze_with_synonym_unification(text_list, field_name):
    print(f"--- Processing {field_name}... Please wait, this might take a moment ---")

    raw_tokens = []
    # Extract meaningful lemmatized tokens
    for doc in nlp.pipe(text_list, disable=["ner", "parser"]):
        tokens = [
            token.lemma_.lower()
            for token in doc
            if not token.is_stop          # remove stopwords
            and not token.is_punct        # remove punctuation
            and not token.like_url        # remove URLs
            and token.text.strip() != ""  # skip empty tokens
        ]
        raw_tokens.extend(tokens)

    # Collect all unique token types
    unique_words = set(raw_tokens)
    synonym_map = {}

    print(f"Building synonym map for {len(unique_words)} unique words...")

    # Map each token to a canonical synonym
    for word in unique_words:
        synonym_map[word] = get_canonical_word(word)

    # Replace all tokens with their canonical synonym
    unified_tokens = [synonym_map[word] for word in raw_tokens]
    word_freq = Counter(unified_tokens)
    total_unique = len(word_freq)

    # Print main statistics after synonym unification
    print(f"\n=== Final Analysis for: {field_name} (After Synonym Unification) ===")
    print(f"Total Unique Words: {total_unique}")

    print("\nTop 20 Most Common Words (Unified):")
    for word, count in word_freq.most_common(20):
        print(f"{word}: {count}")

    print("\nTop 20 Rarest Words (Unified - Sample):")
    for word, count in word_freq.most_common()[:-21:-1]:
        print(f"{word}: {count}")

    print("=" * 40 + "\n")


# Run synonym-unified analysis if the variables exist
if 'abstracts' in locals():
    analyze_with_synonym_unification(abstracts, "Abstracts")
    analyze_with_synonym_unification(titles, "Titles")


--- Processing Abstracts... Please wait, this might take a moment ---
Building synonym map for 24471 unique words...

=== Final Analysis for: Abstracts (After Synonym Unification) ===
Total Unique Words: 22660

Top 20 Most Common Words (Unified):
protein: 2914
cell: 2856
patient: 2373
consequence: 2302
survey: 1647
gene: 1474
addition: 1404
factor: 1352
show: 1241
associate: 1203
function: 1192
mutant: 1189
degree: 1156
hazard: 1022
group: 1014
control: 961
propose: 945
high: 922
two: 918
activity: 892

Top 20 Rarest Words (Unified - Sample):
arg-70: 1
thr-248: 1
arg-240: 1
gln-236: 1
mgdp: 1
di(tri)phosphate: 1
methylanthraniloyl: 1
cases)--occurrence: 1
hydrolysate: 1
immunostimulating: 1
chemopreventative: 1
extranutritional: 1
glycosylphosphatidyl: 1
deferen: 1
seminal: 1
postvasectomy: 1
periodate: 1
one-: 1
immunocontraceptive: 1
mesenchymal: 1

--- Processing Titles... Please wait, this might take a moment ---
Building synonym map for 7556 unique words...

=== Final Analysis for

For the advanced analysis, we implemented a synonym consolidation method using NLTK's WordNet, a comprehensive lexical database for the English language. This process involved mapping diverse terms to their canonical synonyms - for instance, grouping specific terms like 'carcinoma' under the broader category of 'cancer,' or unifying 'illness' and 'disease.' By leveraging WordNet, we were able to automatically identify and merge semantically equivalent words across the corpus.

The primary advantage of this approach is the significant improvement in Recall, addressing the common 'vocabulary mismatch problem' where a user's query differs lexically from the terms used in relevant documents (e.g., searching for 'heart attack' while the text uses 'myocardial infarction'). Furthermore, this consolidation reduces the dimensionality of the feature space by grouping semantically equivalent terms, thereby creating a more efficient retrieval model that captures the semantic intent of the search rather than relying solely on exact string matching.